<a href="https://colab.research.google.com/github/Jana-HQ/cross-mouse-v1-rsa/blob/main/notebooks/preprocessing_cross_mouse_v1_rsa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing — Mouse V1 Cross-Mouse Consistency
Downloads all 32 `brain_observatory_1.1` sessions from the Allen Brain Observatory,
extracts every raw quantity needed by Parts 1–9, and packages them into a single
compact archive (`v1_preprocessed_data.zip`).

**Run this notebook once.** It is the only notebook in the project that touches
the network or the Allen SDK. Everything after this is array slicing on cached files.

In [ ]:
# @title Install dependencies
!pip install -q --no-deps allensdk
!pip install -q "numpy==1.26.4" "scipy==1.11.4" "pandas>=2.0,<2.3" --force-reinstall
!pip install -q argschema boto3 glymur hdmf ndx-events psycopg2-binary pynrrd pynwb scikit-build semver simpleitk xarray
!pip install -q torchvision pyarrow
!pip install -q "numpy==1.26.4" "scipy==1.11.4" --force-reinstall --no-deps

In [1]:
# @title Imports and global parameters
import os, gc, glob, json, shutil, warnings
import numpy as np
import pandas as pd
import h5py
from PIL import Image

warnings.filterwarnings('ignore')

SEED        = 42
CACHE_DIR   = '/content/data'              # raw AllenSDK cache (deleted at the end)
OUT_DIR     = '/content/preprocessed'      # final package contents
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)

BIN_WIDTH_MS = 1
BIN_EDGES_MS = np.arange(-75, 405, BIN_WIDTH_MS)     # 96 edges -> 95 bins
N_BINS       = len(BIN_EDGES_MS) - 1

N_IMAGES        = 118
MIN_UNITS       = 10
QC_AMP_CUTOFF   = 0.1
QC_PRESENCE_MIN = 0.9
QC_ISI_MAX      = 0.5
AREAS           = ['VISp', 'VISl', 'VISal', 'VISpm', 'VISam']

OPTO_LIGHT_WIN    = 0.005   # 5 ms post-pulse
OPTO_BASELINE_WIN = 0.005   # 5 ms pre-pulse
STATE_WINDOW      = (0.0, 0.25)

print(f'{N_BINS} bins of {BIN_WIDTH_MS} ms, spanning '
      f'[{BIN_EDGES_MS[0]}, {BIN_EDGES_MS[-1]}] ms relative to stimulus onset.')

479 bins of 1 ms, spanning [-75, 404] ms relative to stimulus onset.


In [2]:
# @title Core binning utility (single pass per unit, all bins at once)

def load_unit_spike_times(nwb_path, unit_ids):
    """Return {unit_id: sorted spike-time array} for the requested units."""
    unit_id_set = set(unit_ids)
    ust = {}
    with h5py.File(nwb_path, 'r') as f:
        nwb_ids  = f['units/id'][:]
        spk_flat = f['units/spike_times'][:]
        spk_idx  = f['units/spike_times_index'][:]
        for i, uid in enumerate(nwb_ids):
            if uid not in unit_id_set:
                continue
            s = 0 if i == 0 else int(spk_idx[i - 1])
            e = int(spk_idx[i])
            ust[uid] = spk_flat[s:e]
    return ust

def bin_spikes_per_trial(spike_times, trial_starts, bin_edges_ms=BIN_EDGES_MS):
    n_trials = len(trial_starts)
    n_bins   = len(bin_edges_ms) - 1
    counts   = np.zeros((n_trials, n_bins), dtype=np.int32)
    if len(spike_times) == 0:
        return counts

    dt_lo = bin_edges_ms[0]  / 1000.0   # −0.075 s
    dt_hi = bin_edges_ms[-1] / 1000.0   # +0.400 s

    for i, t0 in enumerate(trial_starts):
        lo = np.searchsorted(spike_times, t0 + dt_lo, side='left')
        hi = np.searchsorted(spike_times, t0 + dt_hi, side='left')
        if lo >= hi:
            continue
        rel_ms  = (spike_times[lo:hi] - t0) * 1000.0
        bidx    = np.clip(np.digitize(rel_ms, bin_edges_ms) - 1, 0, n_bins - 1)
        np.add.at(counts[i], bidx, 1)

    return counts

print('Binning utility defined.')

Binning utility defined.


In [3]:
# @title Load Allen SDK cache and session/unit tables
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

manifest_path = os.path.join(CACHE_DIR, 'manifest.json')
cache         = EcephysProjectCache.from_warehouse(manifest=manifest_path)

sessions_df = cache.get_session_table()
bo_sessions = sessions_df[sessions_df['session_type'] == 'brain_observatory_1.1'].copy()
units_all   = cache.get_units()

# VISp sessions = the 32-session backbone used throughout the paper
visp_units      = units_all[units_all.ecephys_structure_acronym == 'VISp']
visp_counts     = visp_units.groupby('ecephys_session_id').size().sort_values(ascending=False)
bo_session_ids  = set(bo_sessions.index)
visp_counts_bo  = visp_counts[visp_counts.index.isin(bo_session_ids)]
selected_session_ids = visp_counts_bo.index.tolist()

# PV-Cre sessions (optotagging, Part 5.4) — subset of the same 32 sessions
pv_session_ids = bo_sessions[
    bo_sessions.full_genotype.str.contains('Pvalb', na=False)
].index.tolist()
pv_session_ids = [s for s in pv_session_ids if s in selected_session_ids]

# Per-area session lists (Part 9) — sessions with >=10 QC-passing units in that area
def qc_unit_ids(session_id, area):
    sub = units_all[
        (units_all.ecephys_session_id == session_id) &
        (units_all.ecephys_structure_acronym == area) &
        (units_all.amplitude_cutoff < QC_AMP_CUTOFF) &
        (units_all.presence_ratio   > QC_PRESENCE_MIN) &
        (units_all.isi_violations   < QC_ISI_MAX)
    ]
    return sub.index.tolist()

area_session_ids = {}
for area in AREAS:
    sids = [sid for sid in selected_session_ids if len(qc_unit_ids(sid, area)) >= MIN_UNITS]
    area_session_ids[area] = sids
    print(f'{area:>6s}: {len(sids)} sessions with >= {MIN_UNITS} QC-passing units')

print(f'\nVISp backbone sessions : {len(selected_session_ids)}')
print(f'PV-Cre sessions        : {len(pv_session_ids)}')

  VISp: 32 sessions with >= 10 QC-passing units
  VISl: 24 sessions with >= 10 QC-passing units
 VISal: 23 sessions with >= 10 QC-passing units
 VISpm: 20 sessions with >= 10 QC-passing units
 VISam: 26 sessions with >= 10 QC-passing units

VISp backbone sessions : 32
PV-Cre sessions        : 5


In [ ]:
# @title Download all needed NWB files up front
all_needed_sids = sorted(set(selected_session_ids) |
                          set().union(*area_session_ids.values()))

already = [sid for sid in all_needed_sids
           if glob.glob(os.path.join(CACHE_DIR, f'session_{sid}',
                                      f'session_{sid}.nwb'))]
to_download = [sid for sid in all_needed_sids if sid not in already]

print(f'Sessions needed  : {len(all_needed_sids)}')
print(f'Already on disk  : {len(already)}')
print(f'To download      : {len(to_download)}')

for sid in to_download:
    print(f'  Downloading {sid}...', end=' ', flush=True)
    _ = cache.get_session_data(sid)
    gc.collect()
    print('done')

print('All sessions ready.')

In [5]:
# @title Extract natural-scenes binned tensors for all five areas + 0-250 ms trial table

def get_nwb_path(session_id):
    matches = glob.glob(os.path.join(CACHE_DIR, f"session_{session_id}",
                                      f"session_{session_id}.nwb"))
    return matches[0]

def get_ns_presentations(session, n_expected_images=N_IMAGES):
    stim = session.stimulus_presentations.copy()
    stim['frame'] = pd.to_numeric(stim['frame'], errors='coerce')
    ns = stim[(stim.stimulus_name == 'natural_scenes') & (stim['frame'] >= 0)].copy()
    ns = ns.sort_values('start_time').reset_index(drop=True)
    image_ids = sorted(ns.frame.unique())
    if len(image_ids) != n_expected_images:
        return None, None
    return ns, image_ids

h5_path = os.path.join(OUT_DIR, 'responses.h5')
with h5py.File(h5_path, 'w') as h5f:
    for area in AREAS:
        grp = h5f.create_group(f'natural_scenes/{area}')
        for sid in area_session_ids[area]:
            unit_ids = qc_unit_ids(sid, area)
            session  = cache.get_session_data(sid)
            ns, image_ids = get_ns_presentations(session)
            if ns is None:
                del session; gc.collect(); continue
            nwb_path = get_nwb_path(sid)
            ust      = load_unit_spike_times(nwb_path, unit_ids)
            starts   = ns['start_time'].values
            frames   = ns['frame'].values.astype(int)
            tensor = np.zeros((N_IMAGES, len(unit_ids), N_BINS), dtype=np.float32)
            for j, uid in enumerate(unit_ids):
                trial_bins = bin_spikes_per_trial(ust.get(uid, np.array([])), starts)
                for i, img in enumerate(image_ids):
                    tensor[i, j, :] = trial_bins[frames == img].mean(axis=0)
            sgrp = grp.create_group(str(sid))
            sgrp.create_dataset('tensor', data=tensor, compression='gzip', compression_opts=4)
            sgrp.create_dataset('unit_ids', data=np.array(unit_ids))
            sgrp.create_dataset('image_ids', data=np.array(image_ids))
            if area == 'VISp':
                t0, t1 = 0.0, 0.25
                trial_counts = np.zeros((len(starts), len(unit_ids)), dtype=np.float32)
                for j, uid in enumerate(unit_ids):
                    spikes = ust.get(uid, np.array([]))
                    if len(spikes) == 0:
                        continue
                    trial_starts_w = starts + t0
                    trial_ends_w   = starts + t1
                    idx   = np.searchsorted(trial_starts_w, spikes, side='right') - 1
                    valid = (idx >= 0) & (idx < len(starts))
                    idx_v = idx[valid]; sp_v = spikes[valid]
                    in_w  = sp_v < trial_ends_w[idx_v]
                    np.add.at(trial_counts[:, j], idx_v[in_w], 1)
                sgrp.create_dataset('trial_level_0_250ms', data=trial_counts,
                                     compression='gzip', compression_opts=4)
                sgrp.create_dataset('trial_frame', data=frames)
                sgrp.create_dataset('trial_start_time', data=starts)
            del session, ust, tensor; gc.collect()
        print(f'{area}: tensors written for {len(area_session_ids[area])} sessions')

print(f'\nWrote {h5_path}')

VISp: tensors written for 32 sessions
VISl: tensors written for 24 sessions
VISal: tensors written for 23 sessions
VISpm: tensors written for 20 sessions
VISam: tensors written for 26 sessions

Wrote /content/preprocessed/responses.h5


In [ ]:
# @title Extract static-gratings binned tensor
MIN_GRATING_CONDITIONS = 10

def get_grating_presentations(session):
    stim = session.stimulus_presentations.copy()
    sg = stim[stim.stimulus_name == 'static_gratings'].copy()
    sg = sg[sg.orientation.notna() & (sg.orientation != 'null')].copy()
    sg = sg[sg.spatial_frequency.notna() & (sg.spatial_frequency != 'null')].copy()
    sg['condition'] = list(zip(sg['orientation'].astype(float).round(1),
                                sg['spatial_frequency'].astype(float).round(3)))
    return sg.sort_values('start_time').reset_index(drop=True)

# First pass: find conditions common to every VISp session
session_conditions = {}
for sid in selected_session_ids:
    session = cache.get_session_data(sid)
    sg = get_grating_presentations(session)
    session_conditions[sid] = set(sg['condition'].unique())
    del session; gc.collect()

common_conditions = sorted(set.intersection(*session_conditions.values()))
N_GRATING_CONDS   = len(common_conditions)
print(f'Common grating conditions across all sessions: {N_GRATING_CONDS}')

with h5py.File(h5_path, 'a') as h5f:

    if 'static_gratings/VISp' in h5f:
        del h5f['static_gratings/VISp']
    grp = h5f.create_group('static_gratings/VISp')
    grp.create_dataset('conditions', data=np.array(common_conditions))

    for sid in selected_session_ids:
        unit_ids = qc_unit_ids(sid, 'VISp')
        session  = cache.get_session_data(sid)
        sg       = get_grating_presentations(session)
        sg       = sg[sg['condition'].isin(set(common_conditions))].reset_index(drop=True)
        if len(sg) == 0:
            del session; gc.collect(); continue

        nwb_path = get_nwb_path(sid)
        ust      = load_unit_spike_times(nwb_path, unit_ids)
        starts   = sg['start_time'].values
        cond_arr = sg['condition'].values

        tensor = np.zeros((N_GRATING_CONDS, len(unit_ids), N_BINS), dtype=np.float32)
        for j, uid in enumerate(unit_ids):
            trial_bins = bin_spikes_per_trial(ust.get(uid, np.array([])), starts)
            for i, cond in enumerate(common_conditions):
                mask = np.array([c == cond for c in cond_arr])
                tensor[i, j, :] = trial_bins[mask].mean(axis=0)

        sgrp = grp.create_group(str(sid))
        sgrp.create_dataset('tensor', data=tensor, compression='gzip', compression_opts=4)
        sgrp.create_dataset('unit_ids', data=np.array(unit_ids))

        del session, ust, tensor; gc.collect()

print('Gratings tensors written.')


In [7]:
# @title Compute and cache gratings noise ceiling (VISp)
from scipy.stats import spearmanr

N_SPLITS_NC = 20
rng_nc      = np.random.default_rng(SEED)

nc_gr_per_mouse = []   # (n_sessions, n_conditions)

for sid in selected_session_ids:
    unit_ids = qc_unit_ids(sid, 'VISp')
    if len(unit_ids) < MIN_UNITS:
        continue
    session = cache.get_session_data(sid)
    sg      = get_grating_presentations(session)
    sg      = sg[sg['condition'].isin(set(common_conditions))].reset_index(drop=True)
    if len(sg) == 0:
        del session; gc.collect(); continue

    nwb_path = get_nwb_path(sid)
    ust      = load_unit_spike_times(nwb_path, unit_ids)
    starts   = sg['start_time'].values
    cond_arr = sg['condition'].values

    # 0-250 ms trial-level counts — built on the fly, not stored
    trial_counts = np.zeros((len(starts), len(unit_ids)), dtype=np.float32)
    for j, uid in enumerate(unit_ids):
        spikes = ust.get(uid, np.array([]))
        if len(spikes) == 0:
            continue
        ts = starts + 0.0; te = starts + 0.25
        idx   = np.searchsorted(ts, spikes, side='right') - 1
        valid = (idx >= 0) & (idx < len(starts))
        idx_v = idx[valid]; sp_v = spikes[valid]
        in_w  = sp_v < te[idx_v]
        np.add.at(trial_counts[:, j], idx_v[in_w], 1)

    reliab = np.zeros((N_SPLITS_NC, N_GRATING_CONDS))
    for s in range(N_SPLITS_NC):
        for i, cond in enumerate(common_conditions):
            idx = np.where([c == cond for c in cond_arr])[0]
            if len(idx) < 4:
                reliab[s, i] = np.nan; continue
            rng_nc.shuffle(idx); half = len(idx) // 2
            a = trial_counts[idx[:half]].mean(0)
            b = trial_counts[idx[half:2*half]].mean(0)
            if a.std() == 0 or b.std() == 0:
                reliab[s, i] = np.nan; continue
            r, _ = spearmanr(a, b)
            reliab[s, i] = r
    mean_r = np.nanmean(reliab, axis=0)
    nc_gr_per_mouse.append((2 * mean_r) / (1 + mean_r))
    del session, ust, trial_counts; gc.collect()

nc_gr_arr = np.array(nc_gr_per_mouse)   # (n_sessions, n_conditions)
np.save(os.path.join(OUT_DIR, 'noise_ceiling_gratings_VISp.npy'), nc_gr_arr)
print(f'Gratings NC — mean: {np.nanmean(nc_gr_arr):.3f},  '
      f'range: [{np.nanmin(nc_gr_arr):.3f}, {np.nanmax(nc_gr_arr):.3f}]')

Gratings NC — mean: 0.992,  range: [0.949, 0.998]


In [ ]:
# @title Compute and cache noise ceilings for non-VISp areas

NC_AREAS = ['VISl', 'VISal', 'VISpm', 'VISam']

with h5py.File(h5_path, 'a') as h5f:
    for area in NC_AREAS:
        nc_key = f'natural_scenes/{area}/noise_ceiling'
        if nc_key in h5f:
            del h5f[nc_key]
        nc_rows = []

        sids_area = area_session_ids[area]
        for sid in sids_area:
            unit_ids = qc_unit_ids(sid, area)
            if len(unit_ids) < MIN_UNITS:
                continue
            session = cache.get_session_data(sid)
            ns, image_ids = get_ns_presentations(session)
            if ns is None:
                del session; gc.collect(); continue

            nwb_path = get_nwb_path(sid)
            ust      = load_unit_spike_times(nwb_path, unit_ids)
            starts   = ns['start_time'].values
            frames   = ns['frame'].values.astype(int)

            trial_counts = np.zeros((len(starts), len(unit_ids)), dtype=np.float32)
            for j, uid in enumerate(unit_ids):
                spikes = ust.get(uid, np.array([]))
                if len(spikes) == 0:
                    continue
                ts = starts; te = starts + 0.25
                idx   = np.searchsorted(ts, spikes, side='right') - 1
                valid = (idx >= 0) & (idx < len(starts))
                idx_v = idx[valid]; sp_v = spikes[valid]
                in_w  = sp_v < te[idx_v]
                np.add.at(trial_counts[:, j], idx_v[in_w], 1)

            rng_nc2 = np.random.default_rng(SEED)
            reliab  = np.zeros((N_SPLITS_NC, N_IMAGES))
            for s in range(N_SPLITS_NC):
                for i, img in enumerate(image_ids):
                    idx = np.where(frames == img)[0]
                    if len(idx) < 4:
                        reliab[s, i] = np.nan; continue
                    rng_nc2.shuffle(idx); half = len(idx) // 2
                    a = trial_counts[idx[:half]].mean(0)
                    b = trial_counts[idx[half:2*half]].mean(0)
                    if a.std() == 0 or b.std() == 0:
                        reliab[s, i] = np.nan; continue
                    r, _ = spearmanr(a, b)
                    reliab[s, i] = r
            mean_r = np.nanmean(reliab, axis=0)
            sb     = (2 * mean_r) / (1 + mean_r)
            nc_rows.append(sb)
            del session, ust, trial_counts; gc.collect()

        nc_arr = np.array(nc_rows)   # (n_sessions_in_area, n_images)
        h5f[f'natural_scenes/{area}'].create_dataset(
            'noise_ceiling', data=nc_arr, compression='gzip', compression_opts=4)
        print(f'{area}: NC mean = {np.nanmean(nc_arr):.3f}  '
              f'(n = {len(nc_rows)} sessions)')

In [ ]:
# @title Pupil area and running speed

KNOWN_NO_EYE = {732592105, 715093703, 719161530, 721123822, 737581020, 739448407}

def read_pupil_from_nwb(nwb_path):
    """
    Read pupil area and timestamps directly from NWB file, bypassing AllenSDK.
    Uses processing/filtered_gaze_mapping/pupil_area — the filtered (smoothed)
    pupil trace, equivalent to what get_pupil_data() returned before pynwb
    removed ProcessingModule.get_data_interface().
    Returns (timestamps, area) as float64 arrays, or (None, None) if absent.
    """
    KEY_DATA = 'processing/filtered_gaze_mapping/pupil_area/data'
    KEY_TIME = 'processing/filtered_gaze_mapping/pupil_area/timestamps'
    with h5py.File(nwb_path, 'r') as f:
        if KEY_DATA not in f or KEY_TIME not in f:
            return None, None
        return f[KEY_TIME][:].astype(float), f[KEY_DATA][:].astype(float)

state_records = []

for sid in selected_session_ids:
    session = cache.get_session_data(sid)
    ns, image_ids = get_ns_presentations(session)
    if ns is None:
        del session; gc.collect(); continue

    # ── Pupil: direct NWB read ───────────────────────────────────────────────
    nwb_path = get_nwb_path(sid)
    if sid not in KNOWN_NO_EYE:
        pupil_times, pupil_area_vals = read_pupil_from_nwb(nwb_path)
    else:
        pupil_times, pupil_area_vals = None, None

    # ── Running speed: SDK still works for this ──────────────────────────────
    running = None
    try:
        running = session.running_speed
    except Exception:
        pass

    for _, row in ns.iterrows():
        t0 = row['start_time'] + STATE_WINDOW[0]
        t1 = row['start_time'] + STATE_WINDOW[1]
        rec = {'session_id': sid, 'image_id': int(row['frame'])}

        if pupil_times is not None:
            m = (pupil_times >= t0) & (pupil_times < t1)
            rec['pupil_area'] = float(np.nanmean(pupil_area_vals[m])) \
                                 if m.sum() > 0 else np.nan
        else:
            rec['pupil_area'] = np.nan

        if running is not None and len(running) > 0:
            m = ((running['start_time'].values < t1) &
                 (running['end_time'].values   > t0))
            rec['running_speed'] = float(np.nanmean(running['velocity'].values[m])) \
                                    if m.sum() > 0 else np.nan
        else:
            rec['running_speed'] = np.nan

        state_records.append(rec)

    del session; gc.collect()

df_state = pd.DataFrame(state_records)

# Quick sanity print before saving
n_eye = df_state.groupby('session_id').pupil_area.apply(
    lambda x: x.notna().any()).sum()
n_run = df_state.groupby('session_id').running_speed.apply(
    lambda x: x.notna().any()).sum()
print(f'Presentations : {len(df_state)}')
print(f'Sessions      : {df_state.session_id.nunique()}')
print(f'With pupil    : {n_eye} / {len(selected_session_ids)} '
      f'(expect 26; {len(KNOWN_NO_EYE)} known missing)')
print(f'With running  : {n_run} / {len(selected_session_ids)} (expect 32)')
print()

# Spot-check one session that should have eye tracking
test_sid = next(s for s in selected_session_ids if s not in KNOWN_NO_EYE)
sub = df_state[df_state.session_id == test_sid]
print(f'Session {test_sid} — '
      f'pupil non-NaN: {sub.pupil_area.notna().sum()}/{len(sub)}, '
      f'mean pupil area: {sub.pupil_area.mean():.1f}')

In [10]:
# @title Optotagging: per-unit light/baseline pulse counts
opto_records = []

def read_opto_pulse_starts(nwb_path):
    BASE = 'processing/optotagging/optogenetic_stimulation'
    with h5py.File(nwb_path, 'r') as f:
        if BASE not in f:
            return None
        raw = f[f'{BASE}/stimulus_name'][:]
        stim_names = np.array([
            s.decode('utf-8') if isinstance(s, bytes) else str(s)
            for s in raw
        ])
        pulse_mask = stim_names == 'pulse'
        if pulse_mask.sum() < 10:
            return None
        return f[f'{BASE}/start_time'][:][pulse_mask]

for sid in pv_session_ids:
    units_s = units_all[
        (units_all.ecephys_session_id        == sid) &
        (units_all.ecephys_structure_acronym == 'VISp') &
        (units_all.amplitude_cutoff < QC_AMP_CUTOFF) &
        (units_all.presence_ratio   > QC_PRESENCE_MIN) &
        (units_all.isi_violations   < QC_ISI_MAX)
    ]
    nwb_path     = get_nwb_path(sid)
    pulse_starts = read_opto_pulse_starts(nwb_path)
    if pulse_starts is None:
        print(f'Session {sid}: no usable pulse epochs — skipping')
        continue

    ust = load_unit_spike_times(nwb_path, units_s.index.tolist())
    for uid in units_s.index:
        spikes = ust.get(uid, np.array([]))
        light  = np.mean([((spikes >= t) & (spikes < t + OPTO_LIGHT_WIN)).sum()
                           for t in pulse_starts])
        base   = np.mean([((spikes >= t - OPTO_BASELINE_WIN) & (spikes < t)).sum()
                           for t in pulse_starts])
        opto_records.append({
            'session_id': sid, 'unit_id': uid,
            'opto_light_mean': light, 'opto_baseline_mean': base,
        })
    del ust; gc.collect()
    print(f'Session {sid}: {len(units_s)} units, {len(pulse_starts)} pulses')

df_opto = pd.DataFrame(opto_records)
print(f'\nOptotagging pulse counts extracted for {len(df_opto)} VISp units '
      f'across {len(pv_session_ids)} PV-Cre sessions.')
print(df_opto.head(3).to_string())

Session 721123822: 41 units, 90 pulses
Session 746083955: 14 units, 90 pulses
Session 760345702: 72 units, 90 pulses
Session 773418906: 37 units, 90 pulses
Session 797828357: 85 units, 150 pulses

Optotagging pulse counts extracted for 249 VISp units across 5 PV-Cre sessions.
   session_id    unit_id  opto_light_mean  opto_baseline_mean
0   721123822  950908404         0.033333            0.011111
1   721123822  950908410         0.155556            0.100000
2   721123822  950908414         0.088889            0.100000


In [ ]:
# @title Build global unit metadata table (QC fields, waveform, opto columns)
units_meta = units_all[units_all.ecephys_structure_acronym.isin(AREAS)].copy()
units_meta = units_meta[
    (units_meta.amplitude_cutoff < QC_AMP_CUTOFF) &
    (units_meta.presence_ratio   > QC_PRESENCE_MIN) &
    (units_meta.isi_violations   < QC_ISI_MAX)
].copy()
units_meta = units_meta[['ecephys_session_id', 'ecephys_structure_acronym',
                          'waveform_duration', 'amplitude_cutoff',
                          'presence_ratio', 'isi_violations']].reset_index()
units_meta = units_meta.rename(columns={
    'id': 'unit_id',
    'ecephys_structure_acronym': 'area',
})

assert 'unit_id' in units_meta.columns, f"Expected 'unit_id', got: {units_meta.columns.tolist()}"
assert 'unit_id' in df_opto.columns,    f"df_opto empty or missing unit_id: {df_opto.columns.tolist()}"

units_meta = units_meta.merge(
    df_opto[['session_id', 'unit_id', 'opto_light_mean', 'opto_baseline_mean']]
        .rename(columns={'session_id': 'ecephys_session_id'}),
    on=['ecephys_session_id', 'unit_id'],
    how='left',
)

print(f'Unit metadata: {len(units_meta)} QC-passing units across {len(AREAS)} areas.')
print(f'Units with opto data: {units_meta["opto_light_mean"].notna().sum()}')
units_meta.to_parquet(os.path.join(OUT_DIR, 'units_meta.parquet'), index=False)

In [ ]:
# @title Natural-scene image templates (downsampled) + ResNet50 embeddings
import torch
import torchvision.models as tv_models
import torchvision.transforms as T

natural_scenes_list = [cache.get_natural_scene_template(i) for i in range(N_IMAGES)]
natural_scenes_full = np.stack(natural_scenes_list, axis=0)

# Downsampled copy for gallery figures only — full Allen resolution is overkill
# for an 8-panel figure and triples the archive size for no benefit.
GALLERY_MAX_DIM = 256
templates_small = np.zeros((N_IMAGES, GALLERY_MAX_DIM, GALLERY_MAX_DIM), dtype=np.uint8)
for i, img in enumerate(natural_scenes_full):
    im = Image.fromarray(img.astype(np.uint8))
    im = im.resize((GALLERY_MAX_DIM, GALLERY_MAX_DIM), Image.BILINEAR)
    templates_small[i] = np.array(im)
np.save(os.path.join(OUT_DIR, 'image_templates.npy'), templates_small)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
preprocess = T.Compose([
    T.ToPILImage(), T.Resize(224), T.CenterCrop(224),
    T.Grayscale(num_output_channels=3), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
model = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2).to(device).eval()
activations = {}
def hook(name):
    def fn(m, inp, out): activations[name] = out.detach()
    return fn
layers = {'layer1': model.layer1, 'layer2': model.layer2,
          'layer3': model.layer3, 'layer4': model.layer4}
handles = [l.register_forward_hook(hook(n)) for n, l in layers.items()]

emb_by_layer = {n: [] for n in layers}; emb_final = []
with torch.no_grad():
    for img in natural_scenes_full:
        img_u8 = ((img - img.min()) / (img.max() - img.min() + 1e-8) * 255).astype(np.uint8)
        x = preprocess(img_u8).unsqueeze(0).to(device)
        out = model(x)
        emb_final.append(out.cpu().numpy().flatten())
        for n in layers:
            emb_by_layer[n].append(activations[n].mean(dim=[2, 3]).cpu().numpy().flatten())
for h in handles: h.remove()

np.savez(os.path.join(OUT_DIR, 'cnn_embeddings.npz'),
         layer1=np.stack(emb_by_layer['layer1']),
         layer2=np.stack(emb_by_layer['layer2']),
         layer3=np.stack(emb_by_layer['layer3']),
         layer4=np.stack(emb_by_layer['layer4']),
         final=np.stack(emb_final))

del natural_scenes_full, natural_scenes_list, model; gc.collect()
print('Image templates and CNN embeddings saved.')

In [ ]:
# @title Save behavioral-state table and write manifest.json
df_state.to_parquet(os.path.join(OUT_DIR, 'behavioral_state.parquet'), index=False)

manifest = {
    'bin_width_ms':        BIN_WIDTH_MS,
    'bin_edges_ms':        BIN_EDGES_MS.tolist(),
    'n_images':            N_IMAGES,
    'n_grating_conditions': N_GRATING_CONDS,
    'grating_conditions':  [list(c) for c in common_conditions],
    'areas':               AREAS,
    'selected_session_ids_visp': selected_session_ids,
    'area_session_ids':    area_session_ids,
    'pv_cre_session_ids':  pv_session_ids,
    'qc_thresholds': {
        'amplitude_cutoff_max': QC_AMP_CUTOFF,
        'presence_ratio_min':   QC_PRESENCE_MIN,
        'isi_violations_max':   QC_ISI_MAX,
        'min_units_per_session': MIN_UNITS,
    },
    'opto_windows_s': {'light': OPTO_LIGHT_WIN, 'baseline': OPTO_BASELINE_WIN},
    'state_window_s': list(STATE_WINDOW),
    'seed': SEED,

}
with open(os.path.join(OUT_DIR, 'manifest.json'), 'w') as f:
    json.dump(manifest, f, indent=2)

print('manifest.json written.')
print(json.dumps({k: v for k, v in manifest.items()
                   if k not in ('bin_edges_ms', 'grating_conditions')}, indent=2))

In [16]:
# @title Validation — check all extracted objects before zipping
import json
import numpy as np
import pandas as pd
import h5py

PASS = '✅'; FAIL = '❌'
issues = []

def check(label, cond, detail=''):
    status = PASS if cond else FAIL
    if not cond:
        issues.append(label)
    print(f'  {status} {label}' + (f'  [{detail}]' if detail else ''))

print('━━━ manifest.json ━━━')
with open(os.path.join(OUT_DIR, 'manifest.json')) as f:
    mf = json.load(f)
check('has bin_edges_ms',       'bin_edges_ms' in mf)
check('bin_edges_ms length',    len(mf['bin_edges_ms']) == N_BINS + 1,
      f'{len(mf["bin_edges_ms"])} edges')
check('has all areas',          set(mf['areas']) == set(AREAS))
check('visp session count',     len(mf['selected_session_ids_visp']) == 32,
      str(len(mf['selected_session_ids_visp'])))
check('pv session count',       len(mf['pv_cre_session_ids']) == 5,
      str(len(mf['pv_cre_session_ids'])))
check('grating cond count',     mf['n_grating_conditions'] == N_GRATING_CONDS,
      str(mf['n_grating_conditions']))

expected_area_counts = {a: len(area_session_ids[a]) for a in AREAS}

with h5py.File(h5_path, 'r') as h5f:

    print('\n━━━ responses.h5 — natural scenes ━━━')
    for area in AREAS:
        grp_key = f'natural_scenes/{area}'
        if grp_key in h5f:
            n_sessions = sum(1 for k in h5f[grp_key].keys() if k != 'noise_ceiling')
        else:
            n_sessions = 0
        check(f'{area}: session count',
              n_sessions == expected_area_counts[area],
              f'{n_sessions} / {expected_area_counts[area]}')
        if grp_key not in h5f:
            continue
        sid0 = list(h5f[grp_key].keys())[0]
        t = h5f[f'{grp_key}/{sid0}/tensor']
        check(f'{area}/{sid0}: tensor ndim == 3',   t.ndim == 3, str(t.shape))
        check(f'{area}/{sid0}: n_images == {N_IMAGES}', t.shape[0] == N_IMAGES,
              str(t.shape[0]))
        check(f'{area}/{sid0}: n_bins == {N_BINS}', t.shape[2] == N_BINS,
              str(t.shape[2]))
        check(f'{area}/{sid0}: n_units >= {MIN_UNITS}', t.shape[1] >= MIN_UNITS,
              str(t.shape[1]))
        check(f'{area}/{sid0}: no NaN', not np.isnan(t[:]).any())
        check(f'{area}/{sid0}: non-zero entries exist', t[:].sum() > 0)

    print('\n━━━ responses.h5 — VISp trial-level table ━━━')
    visp_grp = h5f['natural_scenes/VISp']
    n_trial_level = sum(1 for sid in visp_grp
                        if 'trial_level_0_250ms' in visp_grp[sid])
    check('all VISp sessions have trial_level table',
          n_trial_level == expected_area_counts['VISp'],
          f'{n_trial_level} / {expected_area_counts["VISp"]}')
    sid0 = list(visp_grp.keys())[0]
    tl = visp_grp[f'{sid0}/trial_level_0_250ms'][:]
    tf = visp_grp[f'{sid0}/trial_frame'][:]
    ts = visp_grp[f'{sid0}/trial_start_time'][:]
    check('trial_level shape consistent (trials × units)',
          tl.shape[1] == visp_grp[f'{sid0}/tensor'].shape[1],
          f'trial_level {tl.shape}, tensor units {visp_grp[f"{sid0}/tensor"].shape[1]}')
    check('trial_frame length matches trial_level rows',
          len(tf) == tl.shape[0], f'{len(tf)} vs {tl.shape[0]}')
    check('trial_start_time length matches trial_level rows',
          len(ts) == tl.shape[0], f'{len(ts)} vs {tl.shape[0]}')
    check('trial_frame contains expected image ids',
          set(tf).issubset(set(range(N_IMAGES))),
          f'{len(set(tf))} unique frames')
    check('trial_level no NaN', not np.isnan(tl).any())
    check('trial_level non-zero', tl.sum() > 0)

    print('\n━━━ responses.h5 — static gratings ━━━')
    check('static_gratings/VISp group exists', 'static_gratings/VISp' in h5f)
    if 'static_gratings/VISp' in h5f:
        sg_grp = h5f['static_gratings/VISp']
        n_sg_sessions = len([k for k in sg_grp.keys() if k != 'conditions'])
        check('grating session count == 32', n_sg_sessions == 32, str(n_sg_sessions))
        check('conditions dataset exists', 'conditions' in sg_grp)
        sid0 = [k for k in sg_grp.keys() if k != 'conditions'][0]
        t = sg_grp[f'{sid0}/tensor'][:]
        check('grating tensor ndim == 3', t.ndim == 3, str(t.shape))
        check(f'grating n_conditions == {N_GRATING_CONDS}',
              t.shape[0] == N_GRATING_CONDS, str(t.shape[0]))
        check(f'grating n_bins == {N_BINS}', t.shape[2] == N_BINS, str(t.shape[2]))
        check('grating tensor no NaN', not np.isnan(t).any())
        check('grating tensor non-zero', t.sum() > 0)

    print('\n━━━ responses.h5 — non-VISp area noise ceilings ━━━')
    for area in NC_AREAS:
        nc_key = f'natural_scenes/{area}/noise_ceiling'
        check(f'{area}: noise_ceiling dataset exists', nc_key in h5f, f'looked for {nc_key}')
        if nc_key not in h5f:
            continue
        nc = h5f[nc_key][:]
        expected_n = expected_area_counts[area]
        check(f'{area}: NC row count matches session count',
              nc.shape[0] == expected_n, f'{nc.shape[0]} rows / {expected_n} sessions')
        check(f'{area}: NC has {N_IMAGES} columns', nc.shape[1] == N_IMAGES, str(nc.shape[1]))
        check(f'{area}: NC values in [0, 1]',
              np.nanmin(nc) >= -0.05 and np.nanmax(nc) <= 1.0,
              f'[{np.nanmin(nc):.3f}, {np.nanmax(nc):.3f}]')
        check(f'{area}: NC mean > 0.8', np.nanmean(nc) > 0.8, f'mean = {np.nanmean(nc):.3f}')
        frac_nan = np.isnan(nc).mean()
        check(f'{area}: NC fraction NaN < 0.05', frac_nan < 0.05, f'{frac_nan:.3f}')
        rows_all_nan = np.isnan(nc).all(axis=1).sum()
        check(f'{area}: no all-NaN sessions in NC', rows_all_nan == 0, f'{rows_all_nan} rows')

# ── h5f closed here; everything below is independent of it ───────────────────

print('\n━━━ units_meta.parquet ━━━')
um = pd.read_parquet(os.path.join(OUT_DIR, 'units_meta.parquet'))
check('non-empty', len(um) > 0, str(len(um)))
check('required columns present',
      {'unit_id','ecephys_session_id','area','waveform_duration',
       'opto_light_mean','opto_baseline_mean'}.issubset(um.columns),
      str(um.columns.tolist()))
check('all 5 areas present', set(um.area.unique()) == set(AREAS), str(sorted(um.area.unique())))
check('waveform_duration no all-NaN',
      um.waveform_duration.notna().sum() > 0, f'{um.waveform_duration.notna().sum()} non-null')
check('opto cols present for PV sessions',
      um[um.ecephys_session_id.isin(pv_session_ids)].opto_light_mean.notna().any())
check('no duplicate unit rows',
      um.duplicated(['ecephys_session_id','unit_id']).sum() == 0,
      f'{um.duplicated(["ecephys_session_id","unit_id"]).sum()} dupes')

print('\n━━━ cnn_embeddings.npz ━━━')
cnn = np.load(os.path.join(OUT_DIR, 'cnn_embeddings.npz'))
for layer in ['layer1','layer2','layer3','layer4','final']:
    arr = cnn[layer]
    check(f'{layer}: n_images == {N_IMAGES}', arr.shape[0] == N_IMAGES, str(arr.shape))
    check(f'{layer}: no NaN', not np.isnan(arr).any())
    check(f'{layer}: non-constant (std > 0)', arr.std(axis=0).mean() > 0)

print('\n━━━ image_templates.npy ━━━')
tmpl = np.load(os.path.join(OUT_DIR, 'image_templates.npy'))
check('shape (118, 256, 256)', tmpl.shape == (N_IMAGES, 256, 256), str(tmpl.shape))
check('dtype uint8', tmpl.dtype == np.uint8, str(tmpl.dtype))
check('non-zero pixels', tmpl.sum() > 0)
check('value range [0,255]', tmpl.min() >= 0 and tmpl.max() <= 255,
      f'[{tmpl.min()}, {tmpl.max()}]')

print('\n━━━ behavioral_state.parquet ━━━')
bs = pd.read_parquet(os.path.join(OUT_DIR, 'behavioral_state.parquet'))
check('non-empty', len(bs) > 0, str(len(bs)))
check('required columns',
      {'session_id','image_id','pupil_area','running_speed'}.issubset(bs.columns))
check('all 32 sessions present', bs.session_id.nunique() == 32, str(bs.session_id.nunique()))
check('image_id range valid', bs.image_id.between(0, N_IMAGES - 1).all(),
      f'[{bs.image_id.min()}, {bs.image_id.max()}]')
check('running_speed has no all-NaN sessions',
      bs.groupby('session_id').running_speed.apply(lambda x: x.notna().any()).all())
n_eye = bs.groupby('session_id').pupil_area.apply(lambda x: x.notna().any()).sum()
check('pupil_area present in >= 26 sessions (6 known missing)',
      n_eye >= 26, f'{n_eye} sessions with eye data')

print('\n━━━ noise_ceiling_gratings_VISp.npy ━━━')
nc_gr_path = os.path.join(OUT_DIR, 'noise_ceiling_gratings_VISp.npy')
check('file exists', os.path.exists(nc_gr_path), nc_gr_path)
if os.path.exists(nc_gr_path):
    nc_gr = np.load(nc_gr_path)
    check('shape[0] == 32 sessions', nc_gr.shape[0] == 32, f'shape = {nc_gr.shape}')
    check(f'shape[1] == {N_GRATING_CONDS} conditions',
          nc_gr.shape[1] == N_GRATING_CONDS, f'shape = {nc_gr.shape}')
    check('values in [0, 1]', np.nanmin(nc_gr) >= -0.05 and np.nanmax(nc_gr) <= 1.0,
          f'[{np.nanmin(nc_gr):.3f}, {np.nanmax(nc_gr):.3f}]')
    check('mean > 0.9', np.nanmean(nc_gr) > 0.9, f'mean = {np.nanmean(nc_gr):.3f}')
    frac_nan_gr = np.isnan(nc_gr).mean()
    check('fraction NaN < 0.05', frac_nan_gr < 0.05, f'{frac_nan_gr:.3f}')
    rows_all_nan_gr = np.isnan(nc_gr).all(axis=1).sum()
    check('no all-NaN sessions', rows_all_nan_gr == 0, f'{rows_all_nan_gr} rows')
    check('NC mean > raw grating consistency (0.720)',
          np.nanmean(nc_gr) > 0.720, f'NC mean = {np.nanmean(nc_gr):.3f}')

print('\n━━━ archive size estimate ━━━')
total = 0
for root, _, files in os.walk(OUT_DIR):
    for fn in files:
        p = os.path.join(root, fn)
        sz = os.path.getsize(p) / 1e6
        total += sz
        print(f'  {os.path.relpath(p, OUT_DIR):45s} {sz:7.1f} MB')
print(f'  {"TOTAL":45s} {total:7.1f} MB')

print('\n━━━ Summary ━━━')
if issues:
    print(f'{FAIL} {len(issues)} check(s) failed:')
    for iss in issues:
        print(f'   • {iss}')
else:
    print(f'{PASS} All checks passed — safe to zip.')

━━━ manifest.json ━━━
  ✅ has bin_edges_ms
  ✅ bin_edges_ms length  [480 edges]
  ✅ has all areas
  ✅ visp session count  [32]
  ✅ pv session count  [5]
  ✅ grating cond count  [30]

━━━ responses.h5 — natural scenes ━━━
  ✅ VISp: session count  [32 / 32]
  ✅ VISp/715093703: tensor ndim == 3  [(118, 60, 479)]
  ✅ VISp/715093703: n_images == 118  [118]
  ✅ VISp/715093703: n_bins == 479  [479]
  ✅ VISp/715093703: n_units >= 10  [60]
  ✅ VISp/715093703: no NaN
  ✅ VISp/715093703: non-zero entries exist
  ✅ VISl: session count  [24 / 24]
  ✅ VISl/715093703: tensor ndim == 3  [(118, 42, 479)]
  ✅ VISl/715093703: n_images == 118  [118]
  ✅ VISl/715093703: n_bins == 479  [479]
  ✅ VISl/715093703: n_units >= 10  [42]
  ✅ VISl/715093703: no NaN
  ✅ VISl/715093703: non-zero entries exist
  ✅ VISal: session count  [23 / 23]
  ✅ VISal/721123822: tensor ndim == 3  [(118, 37, 479)]
  ✅ VISal/721123822: n_images == 118  [118]
  ✅ VISal/721123822: n_bins == 479  [479]
  ✅ VISal/721123822: n_units >= 1

In [ ]:
# @title Zip the package and trigger download
zip_path = '/content/preprocessed_data'
shutil.make_archive(zip_path, 'zip', OUT_DIR)
full_zip = zip_path + '.zip'

size_mb = os.path.getsize(full_zip) / 1e6
print(f'Archive size: {size_mb:.1f} MB')

for root, _, files in os.walk(OUT_DIR):
    for fn in files:
        p = os.path.join(root, fn)
        print(f'  {os.path.relpath(p, OUT_DIR):45s} {os.path.getsize(p)/1e6:8.2f} MB')

from google.colab import files
files.download(full_zip)